# El bueno

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
import difflib
from datetime import datetime

COLORES_CONOCIDOS = [
    "NEGRO", "NEGRO CARBON", "NEGRO MEDIANOCHE", "NEGRO TITANIO", "NEGRO MATE",
    "BLANCO", "BLANCO ESTRELLA", "BLANCO PERLA", "BLANCO TITANIO",
    "GRIS", "GRIS TITANIO", "GRIS ESPACIAL", "GRIS PERLA", "GRIS GRAFITO", "GRIS PLATA",
    "PLATA", "PLATEADO", "PLATA TITANIO",
    "DORADO", "ORO", "DORADO TITANIO", "CHAMPAGNE",
    "AZUL", "AZUL CLARO", "AZUL OBSCURO", "AZUL OSCURO", "AZUL MARINO",
    "AZUL TITANIO", "AZUL NOCHE", "AZUL HIELO", "AZUL CIELO", "AZUL COBALTO",
    "AZUL REAL", "AZUL ZAFIRO", "AZUL PROFUNDO", "AZUL MEDIANOCHE", "AZUL OCEANO",
    "ROJO", "ROJO OSCURO", "ROJO CORAL", "ROJO TITANIO",
    "VERDE", "VERDE OLIVA", "VERDE MENTA", "VERDE SALVIA", "VERDE TITANIO",
    "VERDE CLARO", "VERDE OSCURO", "VERDE BOSQUE",
    "ROSA", "ROSA CLARO", "ROSA MELOCOTON", "ROSA PERLA", "ROSA TITANIO",
    "MORADO", "MORADO OSCURO", "VIOLETA", "VIOLET", "LILA", "LAVANDA", "PURPURA",
    "NARANJA", "SALMON", "CORAL",
    "AMARILLO", "CAFE", "MARRON", "BEIGE", "CREMA", "DESIERTO",
    "TURQUESA", "CYAN", "AQUA", "MENTA",
    "INDIGO", "ULTRAMARINO", "TITANIO", "NATURAL", "MEDIA NOCHE",
]

def completar_color(color_truncado):
    color_up = color_truncado.upper().strip()
    if not color_up:
        return ""
    if color_up in COLORES_CONOCIDOS:
        return color_up
    prefijo_matches = [c for c in COLORES_CONOCIDOS if c.startswith(color_up)]
    if len(prefijo_matches) == 1:
        return prefijo_matches[0]
    if len(prefijo_matches) > 1:
        return min(prefijo_matches, key=len)
    close = difflib.get_close_matches(color_up, COLORES_CONOCIDOS, n=1, cutoff=0.75)
    if close:
        return close[0]
    return ""

def limpiar_precio_con_rango(texto):
    if not texto:
        return texto
    if ' - ' in texto:
        return texto.split(' - ')[0].strip()
    partes = re.split(r'-(?=\$)', texto)
    if len(partes) >= 2 and partes[0].startswith('$'):
        return partes[0].strip()
    return texto

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    if '$' in str(precio_raw):
        cleaned = re.sub(r'\s+', ' ', precio_raw.strip())
        return limpiar_precio_con_rango(cleaned)
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori_str = precio_normal.split(' - ')[0] if ' - ' in str(precio_normal) else precio_normal
        p_desc_str = precio_descuento.split(' - ')[0] if ' - ' in str(precio_descuento) else precio_descuento
        p_ori = float(re.sub(r'[^\d.]', '', p_ori_str).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', p_desc_str).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def precios_son_rango(normal_text, desc_text):
    try:
        n = float(re.sub(r'[^\d.]', '', normal_text.split(' - ')[0].replace(',', '')))
        d = float(re.sub(r'[^\d.]', '', desc_text.split(' - ')[0].replace(',', '')))
        return n < d
    except Exception:
        return False

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def es_smartwatch(nombre):
    return "smartwatch" in (nombre or "").lower()

def extraer_colores_de_card(product):
    color_picker = product.find("div", attrs={"aria-label": "Color picker"})
    colores = []
    if color_picker:
        for btn in color_picker.find_all("button", attrs={"aria-label": "color"}):
            title = btn.get("title", "")
            if title:
                partes = [p.strip() for p in title.split(",")]
                color = completar_color(partes[-1]) if partes else ""
                capacidad = partes[-2] if len(partes) >= 3 else ""
                colores.append((color, capacidad))
    if not colores:
        colores = [("", "")]
    return colores

def extraer_precios_de_contenedor(element):
    price_container = element.find("div", attrs={"data-testid": re.compile(r"\d+-price$")})
    search_in = price_container if price_container else element
    precio_desc_tag = (
        search_in.find("span", attrs={"data-testid": "discounted"}) or
        search_in.find("span", class_="text-price-primary")
    )
    precio_normal_tag = (
        search_in.find("span", attrs={"data-testid": "original"}) or
        search_in.find("span", attrs={"data-testid": re.compile(r"-price-original-prices$")})
    )
    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
    return precio_normal_tag, precio_desc_tag, precio_normal, precio_desc

def cargar_pagina_con_retry(driver, url, max_intentos=4, espera=10):
    for intento in range(max_intentos):
        try:
            driver.get(url)
            time.sleep(espera + random.uniform(2, 5))
            src = driver.page_source.lower()
            if "upstream connect error" in src or "connection termination" in src or "reset reason" in src:
                wait = 20 * (intento + 1) + random.uniform(5, 15)
                print(f"  [RETRY {intento+1}] Error de conexión, reintentando en {wait:.0f}s...")
                time.sleep(wait)
                continue
            return True
        except Exception as e:
            wait = 20 * (intento + 1)
            print(f"  [RETRY {intento+1}] Excepción: {e}, esperando {wait}s...")
            time.sleep(wait)
    print(f"  [FAIL] No se pudo cargar: {url}")
    return False

def extraer_detalle_producto(url, options):
    resultados = []
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        if not cargar_pagina_con_retry(driver, url, espera=8):
            return resultados
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'details[data-testid*="configurator"]'))
            )
        except Exception:
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-testid*="configurator-price"]'))
                )
            except Exception:
                pass
        time.sleep(1)

        def leer_estado_pagina():
            s = BeautifulSoup(driver.page_source, "html.parser")

            color_name = ""
            color_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-color-picker")})
            if color_det:
                summ = color_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2:
                        color_name = ps[1].text.strip()

            if not color_name:
                image_picker = s.find("div", attrs={"data-testid": "ml-image-picker"})
                if image_picker:
                    for btn in image_picker.find_all("button"):
                        check = btn.find("div", class_=lambda c: c and "bg-prim-500" in c)
                        if check:
                            p_tag = btn.find("p")
                            if p_tag:
                                color_name = p_tag.get_text(strip=True)
                            break

            if not color_name:
                page_text = s.get_text()
                m = re.search(r"Color:\s*([A-Za-z][A-Za-z\s]+?)(?:\s*keyboard|\s*Tama)", page_text)
                if m:
                    color_name = m.group(1).strip()

            size_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-size-picker")})
            size_name = ""
            if size_det:
                summ = size_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2 and "Seleccione" not in ps[1].text:
                        size_name = ps[1].text.strip()
            if not size_name:
                page_text = s.get_text()
                m = re.search(r"Tama.o:\s*(\d+\s*GB)", page_text, re.I)
                if m:
                    size_name = m.group(1).strip()

            price_container = s.find("div", attrs={"data-testid": re.compile(r"\d+-configurator-price$")})
            if price_container:
                precio_desc_tag = price_container.find("span", attrs={"data-testid": "discounted"})
                precio_normal_tag = price_container.find("span", attrs={"data-testid": "original"})
            else:
                precio_desc_tag = (
                    s.find("span", attrs={"data-testid": "discounted"}) or
                    s.find("span", class_="text-price-primary")
                )
                precio_normal_tag = s.find("span", attrs={"data-testid": "original"})
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            if not precio_desc and not precio_normal:
                price_span = s.find("span", class_=re.compile(r"text-heading-2xl"))
                if price_span:
                    precio_desc = price_span.get_text(strip=True)
            return color_name, size_name, precio_normal, precio_desc

        soup = BeautifulSoup(driver.page_source, "html.parser")

        color_picker_div = soup.find("div", attrs={"data-testid": "ml-color-picker"})
        color_testids = []
        if color_picker_div:
            for btn in color_picker_div.find_all("button", attrs={"aria-label": "color"}):
                tid = btn.get("data-testid", "")
                title = btn.get("title", "")
                color_hint = ""
                if title:
                    partes = [p.strip() for p in title.split(",")]
                    color_hint = completar_color(partes[-1]) if partes else ""
                if tid:
                    color_testids.append((tid, color_hint))

        if not color_testids:
            image_picker_div = soup.find("div", attrs={"data-testid": "ml-image-picker"})
            if image_picker_div:
                for btn in image_picker_div.find_all(
                    "button", attrs={"data-testid": re.compile(r"^ml-image-picker-image-\d+$")}
                ):
                    tid = btn.get("data-testid", "")
                    p_tag = btn.find("p")
                    color_hint = p_tag.get_text(strip=True) if p_tag else ""
                    if not color_hint:
                        img = btn.find("img")
                        if img:
                            color_hint = re.sub(r'\s+\d+$', '', img.get("alt", "")).strip()
                    if tid:
                        color_testids.append((tid, color_hint))

        size_rg = soup.find("div", attrs={"role": "radiogroup"})
        size_texts = []
        if size_rg:
            for label in size_rg.find_all("label"):
                span = label.find("span")
                if span and span.text.strip():
                    size_texts.append(span.text.strip())

        combinaciones_unicas = set()

        if not color_testids and not size_texts:
            color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
            codigo = extraer_codigo_producto(driver.current_url)
            sku_var = extraer_sku(driver.current_url)
            clave = (color_name, size_name)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))
        else:
            if not color_testids:
                color_testids = [(None, "")]
            if not size_texts:
                size_texts = [None]

            for color_testid, color_hint in color_testids:
                if color_testid:
                    try:
                        btn = driver.find_element(By.CSS_SELECTOR, f'button[data-testid="{color_testid}"]')
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(1.5)
                    except Exception:
                        pass

                for size_text in size_texts:
                    if size_text:
                        try:
                            label = driver.find_element(
                                By.XPATH, f'//label[.//span[normalize-space()="{size_text}"]]'
                            )
                            driver.execute_script("arguments[0].click();", label)
                            time.sleep(1.5)
                        except Exception:
                            pass

                    color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
                    codigo = extraer_codigo_producto(driver.current_url)
                    sku_var = extraer_sku(driver.current_url)

                    if not color_name and color_hint:
                        color_name = color_hint
                    if not size_name and size_text:
                        size_name = size_text
                    if not precio_desc and not precio_normal:
                        continue

                    clave = (color_name, size_name)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))

    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    finally:
        if driver:
            driver.quit()
    return resultados

def extraer_productos_de_soup(soup):
    return soup.find_all("a", href=re.compile(r"^/tienda/pdp/"))

def procesar_productos(products, fecha_consulta, visited_urls, options):
    data = []
    for product in products:
        marca = product.find("h4")
        nombre = product.find("h3")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        href = product.get("href", "")
        url_cel = "https://www.liverpool.com.mx" + href if href.startswith("/tienda/pdp/") else ""

        precio_normal_tag, precio_desc_tag, precio_normal_text, precio_desc_text = extraer_precios_de_contenedor(product)
        colores = extraer_colores_de_card(product)
        todos_vacios = all(c == "" for c, _ in colores)

        es_solo_desc = (not precio_normal_tag or not precio_normal_text) and precio_desc_tag and precio_desc_text
        es_rango = (
            '-' in precio_normal_text or '-' in precio_desc_text or
            (precio_normal_text and precio_desc_text and precios_son_rango(precio_normal_text, precio_desc_text))
        )
        necesita_visita = url_cel and url_cel not in visited_urls and (es_rango or todos_vacios)

        if necesita_visita:
            visited_urls.add(url_cel)
            time.sleep(random.uniform(5, 12))
            variantes = extraer_detalle_producto(url_cel, options)
            if variantes:
                combinaciones_unicas = set()
                for color_det, codigo_det, sku_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                    clave = (color_det, tamano_det)
                    if clave in combinaciones_unicas:
                        continue
                    combinaciones_unicas.add(clave)
                    if not precio_normal_det and precio_desc_det:
                        precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det))
                        precio_desc_final = "No discount"
                    else:
                        precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_normal_det))
                        precio_desc_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det)) if precio_desc_det else "No discount"
                    sku_url = f"https://www.liverpool.com.mx{href}?skuid={sku_det}" if sku_det else url_cel
                    data.append({
                        "Marca": marca.text.strip() if marca else "",
                        "Nombre": nombre_texto,
                        "Color": color_det,
                        "Capacidad": tamano_det,
                        "CodigoProducto": codigo_det,
                        "SKU": sku_det,
                        "URL": sku_url,
                        "FechaConsulta": fecha_consulta,
                        "Precio_normal": precio_normal_final,
                        "Precio_descuento": precio_desc_final
                    })
            else:
                if es_solo_desc:
                    precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_desc_text))
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_normal_text)) if precio_normal_tag else ""
                    precio_desc_final = limpiar_precio_con_rango(formatear_precio(precio_desc_text)) if precio_desc_tag else "No discount"
                data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": "",
                    "Capacidad": "",
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final
                })
        elif es_solo_desc:
            for color_nombre, capacidad_nombre in colores:
                data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        else:
            for color_nombre, capacidad_nombre in colores:
                data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal_tag else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc_tag else "No discount"
                })
    return data

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

BASE_URL = "https://www.liverpool.com.mx/tienda/N-hn2yreqK8COZekZ8ajJySwGcgqAL7Wnt_UndKuRj95BZQCFVNND4Qg-QZJwWijHi"

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []
visited_urls = set()

driver = webdriver.Chrome(options=options)
if not cargar_pagina_con_retry(driver, BASE_URL, espera=12):
    print("No se pudo cargar la página 1.")
    driver.quit()
else:
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/tienda/pdp/"]'))
        )
    except Exception:
        print("Timeout esperando productos en pagina 1")
    for scroll_pos in range(800, 6000, 800):
        driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    products = extraer_productos_de_soup(soup)
    print(f"Productos encontrados en pagina 1: {len(products)}")
    final_data.extend(procesar_productos(products, fecha_consulta, visited_urls, options))

def scrape_page(page):
    driver = webdriver.Chrome(options=options)
    url_pagina = f"{BASE_URL}/page-{page}"
    if not cargar_pagina_con_retry(driver, url_pagina, espera=12):
        print(f"No se pudo cargar la página {page}.")
        driver.quit()
        return []
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/tienda/pdp/"]'))
        )
    except Exception:
        print(f"Timeout esperando productos en pagina {page}")
    for scroll_pos in range(800, 6000, 800):
        driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    products = extraer_productos_de_soup(soup)
    print(f"Productos encontrados en pagina {page}: {len(products)}")
    return procesar_productos(products, fecha_consulta, visited_urls, options)

for page in range(2, 5):
    final_data.extend(scrape_page(page))

if final_data:
    df = pd.DataFrame(final_data)
    try:
        diccionario = pd.read_csv("diccionario liverpool.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df["SKU"] = df["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df = df.merge(diccionario, on="SKU", how="left")
            df["Capacidad"] = df["Capacidad_y"].combine_first(df["Capacidad_x"])
            df.drop(columns=[col for col in df.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontro 'diccionario liverpool.csv'.")

    df["Porcentaje_descuento"] = df.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )

    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
    ]
    df = df[[c for c in columnas_final if c in df.columns]]
    df = df.drop_duplicates()
    urls_con_color = set(df[df["Color"] != ""]["URL"])
    mask = (df["Color"] == "") & (df["URL"].isin(urls_con_color))
    df = df[~mask]

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_colores_{fecha_hoy} V2.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")
    display(df.head(20))
else:
    print("No se encontraron productos.")


Productos encontrados en pagina 1: 56
Productos encontrados en pagina 2: 56
Productos encontrados en pagina 3: 55
Timeout esperando productos en pagina 4
Productos encontrados en pagina 4: 0
Datos exportados a liverpool_colores_2026-10-23 V2.xlsx


,Marca,Nombre,Color,Capacidad,CodigoProducto,SKU,URL,FechaConsulta,Precio_normal,Precio_descuento,Porcentaje_descuento
0,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Lila medio,128 GB,1181494891,1181495452,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$19,999.00","$12,999.35",35.0%
1,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Azul,128 GB,1181494891,1181495479,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$19,999.00","$12,999.35",35.0%
2,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Negro,128 GB,1181494891,1181495444,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$19,999.00","$12,999.35",35.0%
3,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Negro,256 GB,1181494891,1181494901,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$22,499.00","$14,624.35",35.0%
4,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Amarillo,128 GB,1181494891,1181495461,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$19,999.00","$12,999.35",35.0%
5,GOOGLE PIXEL,Smartphone 10 OLED 6.3 pulgadas,Amarillo,256 GB,1181494891,1181495428,https://www.liverpool.com.mx/tienda/pdp/smartp...,2026-10-23 10:23:32,"$22,499.00","$14,624.35",35.0%
6,SAMSUNG,Galaxy A57 Super AMOLED plus 6.7 pulgadas,Awesome Navy,256 GB,1195303655,1195303688,https://www.liverpool.com.mx/tienda/pdp/galaxy...,2026-10-23 10:23:32,"$11,999.00","$9,999.00",16.67%
7,SAMSUNG,Galaxy A57 Super AMOLED plus 6.7 pulgadas,Awesome Lilac,256 GB,1195303655,1195303696,https://www.liverpool.com.mx/tienda/pdp/galaxy...,2026-10-23 10:23:32,"$11,999.00","$9,999.00",16.67%
8,SAMSUNG,Galaxy A57 Super AMOLED plus 6.7 pulgadas,Awesome Gray,256 GB,1195303655,1195303661,https://www.liverpool.com.mx/tienda/pdp/galaxy...,2026-10-23 10:23:32,"$11,999.00","$9,999.00",16.67%
9,SAMSUNG,Galaxy A57 Super AMOLED plus 6.7 pulgadas,Awesome Icyblue,256 GB,1195303655,1195303670,https://www.liverpool.com.mx/tienda/pdp/galaxy...,2026-10-23 10:23:32,"$12,000.00","$9,599.00",20.01%


# El bueno - IOT

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
import difflib
from datetime import datetime

COLORES_CONOCIDOS = [
    "NEGRO", "NEGRO CARBON", "NEGRO MEDIANOCHE", "NEGRO TITANIO", "NEGRO MATE",
    "BLANCO", "BLANCO ESTRELLA", "BLANCO PERLA", "BLANCO TITANIO",
    "GRIS", "GRIS TITANIO", "GRIS ESPACIAL", "GRIS PERLA", "GRIS GRAFITO", "GRIS PLATA",
    "PLATA", "PLATEADO", "PLATA TITANIO",
    "DORADO", "ORO", "DORADO TITANIO", "CHAMPAGNE",
    "AZUL", "AZUL CLARO", "AZUL OBSCURO", "AZUL OSCURO", "AZUL MARINO",
    "AZUL TITANIO", "AZUL NOCHE", "AZUL HIELO", "AZUL CIELO", "AZUL COBALTO",
    "AZUL REAL", "AZUL ZAFIRO", "AZUL PROFUNDO", "AZUL MEDIANOCHE", "AZUL OCEANO",
    "ROJO", "ROJO OSCURO", "ROJO CORAL", "ROJO TITANIO",
    "VERDE", "VERDE OLIVA", "VERDE MENTA", "VERDE SALVIA", "VERDE TITANIO",
    "VERDE CLARO", "VERDE OSCURO", "VERDE BOSQUE",
    "ROSA", "ROSA CLARO", "ROSA MELOCOTON", "ROSA PERLA", "ROSA TITANIO",
    "MORADO", "MORADO OSCURO", "VIOLETA", "VIOLET", "LILA", "LAVANDA", "PURPURA",
    "NARANJA", "SALMON", "CORAL",
    "AMARILLO", "CAFE", "MARRON", "BEIGE", "CREMA", "DESIERTO",
    "TURQUESA", "CYAN", "AQUA", "MENTA",
    "INDIGO", "ULTRAMARINO", "TITANIO", "NATURAL", "MEDIA NOCHE",
]

def completar_color(color_truncado):
    color_up = color_truncado.upper().strip()
    if not color_up:
        return ""
    if color_up in COLORES_CONOCIDOS:
        return color_up
    prefijo_matches = [c for c in COLORES_CONOCIDOS if c.startswith(color_up)]
    if len(prefijo_matches) == 1:
        return prefijo_matches[0]
    if len(prefijo_matches) > 1:
        return min(prefijo_matches, key=len)
    close = difflib.get_close_matches(color_up, COLORES_CONOCIDOS, n=1, cutoff=0.75)
    if close:
        return close[0]
    return ""

def limpiar_precio_con_rango(texto):
    if not texto:
        return texto
    if ' - ' in texto:
        return texto.split(' - ')[0].strip()
    partes = re.split(r'-(?=\$)', texto)
    if len(partes) >= 2 and partes[0].startswith('$'):
        return partes[0].strip()
    return texto

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    if '$' in str(precio_raw):
        cleaned = re.sub(r'\s+', ' ', precio_raw.strip())
        return limpiar_precio_con_rango(cleaned)
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori_str = precio_normal.split(' - ')[0] if ' - ' in str(precio_normal) else precio_normal
        p_desc_str = precio_descuento.split(' - ')[0] if ' - ' in str(precio_descuento) else precio_descuento
        p_ori = float(re.sub(r'[^\d.]', '', p_ori_str).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', p_desc_str).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_colores_de_card(product):
    color_picker = product.find("div", attrs={"aria-label": "Color picker"})
    colores = []
    if color_picker:
        for btn in color_picker.find_all("button", attrs={"aria-label": "color"}):
            title = btn.get("title", "")
            if title:
                partes = [p.strip() for p in title.split(",")]
                color = completar_color(partes[-1]) if partes else ""
                capacidad = partes[-2] if len(partes) >= 3 else ""
                colores.append((color, capacidad))
    if not colores:
        colores = [("", "")]
    return colores

def extraer_precios_de_contenedor(element):
    price_container = element.find("div", attrs={"data-testid": re.compile(r"\d+-price$")})
    search_in = price_container if price_container else element
    precio_desc_tag = (
        search_in.find("span", attrs={"data-testid": "discounted"}) or
        search_in.find("span", class_="text-price-primary")
    )
    precio_normal_tag = (
        search_in.find("span", attrs={"data-testid": "original"}) or
        search_in.find("span", attrs={"data-testid": re.compile(r"-price-original-prices$")})
    )
    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
    return precio_normal_tag, precio_desc_tag, precio_normal, precio_desc

def cargar_pagina_con_retry(driver, url, max_intentos=4, espera=10):
    for intento in range(max_intentos):
        try:
            driver.get(url)
            time.sleep(espera + random.uniform(2, 5))
            src = driver.page_source.lower()
            if "upstream connect error" in src or "connection termination" in src or "reset reason" in src:
                wait = 20 * (intento + 1) + random.uniform(5, 15)
                print(f"  [RETRY {intento+1}] Error de conexión, reintentando en {wait:.0f}s...")
                time.sleep(wait)
                continue
            return True
        except Exception as e:
            wait = 20 * (intento + 1)
            print(f"  [RETRY {intento+1}] Excepción: {e}, esperando {wait}s...")
            time.sleep(wait)
    print(f"  [FAIL] No se pudo cargar: {url}")
    return False

def extraer_productos_de_soup(soup):
    return soup.find_all("a", href=re.compile(r"^/tienda/pdp/"))

def procesar_productos(products, fecha_consulta):
    data = []
    for product in products:
        marca = product.find("h4")
        nombre = product.find("h3")
        nombre_texto = nombre.text.strip() if nombre else ""
        href = product.get("href", "")
        url_cel = "https://www.liverpool.com.mx" + href if href.startswith("/tienda/pdp/") else ""

        precio_normal_tag, precio_desc_tag, precio_normal_text, precio_desc_text = extraer_precios_de_contenedor(product)
        colores = extraer_colores_de_card(product)
        es_solo_desc = (not precio_normal_tag or not precio_normal_text) and precio_desc_tag and precio_desc_text

        if es_solo_desc:
            for color_nombre, capacidad_nombre in colores:
                data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        else:
            for color_nombre, capacidad_nombre in colores:
                data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal_tag else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc_tag else "No discount"
                })
    return data

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

BASE_URLS = [
    ("https://www.liverpool.com.mx/tienda/catst53117186/N-hn2yreqK8COZekZ8ajJyS74jxSCUiRHPvWd7H8zSjYB_UQT9EaUyn85obyv0JVR8", "Smartwatch"),
    ("https://www.liverpool.com.mx/tienda/catst16778857/N-hn2yreqK8COZekZ8ajJyS-t9cxY3BUM3RtAhunNSbh2ZZtC348dOLfDUXDgezS81", "Headphones"),
]

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []

def scrape_page(base_url, page):
    driver = webdriver.Chrome(options=options)
    url_pagina = base_url if page == 1 else f"{base_url}/page-{page}"
    if not cargar_pagina_con_retry(driver, url_pagina, espera=12):
        print(f"No se pudo cargar la página {page}.")
        driver.quit()
        return []
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/tienda/pdp/"]'))
        )
    except Exception:
        print(f"Timeout esperando productos en pagina {page}")
    for scroll_pos in range(800, 6000, 800):
        driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    products = extraer_productos_de_soup(soup)
    print(f"Productos encontrados en pagina {page}: {len(products)}")
    if not products:
        return None
    return procesar_productos(products, fecha_consulta)

for base_url, categoria in BASE_URLS:
    print(f"=== Scrapeando {categoria}: {base_url} ===")
    for page in range(1, 31):
        filas = scrape_page(base_url, page)
        if filas is None:
            break
        for fila in filas:
            fila["Categoria"] = categoria
        final_data.extend(filas)

if final_data:
    df = pd.DataFrame(final_data)
    df["Capacidad"] = "NA"

    # Completar Color y SKU faltantes desde diccionario2 (hoja Liverpool), cruzando por CodigoProducto
    try:
        dicc = pd.read_excel(r"C:\Users\c90029859\Documents\diccionario2.xlsx", sheet_name="Liverpool", dtype=str, keep_default_na=False)
        dicc = dicc[dicc["Categoria"].isin(["Smartwatch", "Headphones"])]
        dicc["CodigoProducto"] = dicc["CodigoProducto"].str.strip()
        dicc["Color"] = dicc["Color"].str.strip().str.upper()
        dicc["SKU"] = dicc["SKU"].str.strip()
        dicc = dicc[["CodigoProducto", "Color", "SKU"]].drop_duplicates()
        faltan = df["Color"] == ""
        rellenos = df[faltan].drop(columns=["Color", "SKU"]).merge(dicc, on="CodigoProducto", how="left")
        rellenos[["Color", "SKU"]] = rellenos[["Color", "SKU"]].fillna("")
        df = pd.concat([df[~faltan], rellenos], ignore_index=True)
        print(f"Completados desde diccionario2: {(rellenos['Color'] != '').sum()} de {faltan.sum()} sin color")
    except FileNotFoundError:
        print("Advertencia: No se encontro 'diccionario2.xlsx'.")

    df["Porcentaje_descuento"] = df.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )

    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento", "Categoria"
    ]
    df = df[[c for c in columnas_final if c in df.columns]]
    df = df.drop_duplicates()
    urls_con_color = set(df[df["Color"] != ""]["URL"])
    mask = (df["Color"] == "") & (df["URL"].isin(urls_con_color))
    df = df[~mask]

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"Liverpool_IOT_{fecha_hoy}.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")
    display(df.head(20))
else:
    print("No se encontraron productos.")


=== Scrapeando Smartwatch: https://www.liverpool.com.mx/tienda/catst53117186/N-hn2yreqK8COZekZ8ajJyS74jxSCUiRHPvWd7H8zSjYB_UQT9EaUyn85obyv0JVR8 ===
Productos encontrados en pagina 1: 56
Productos encontrados en pagina 2: 13
Timeout esperando productos en pagina 3
Productos encontrados en pagina 3: 0
=== Scrapeando Headphones: https://www.liverpool.com.mx/tienda/catst16778857/N-hn2yreqK8COZekZ8ajJyS-t9cxY3BUM3RtAhunNSbh2ZZtC348dOLfDUXDgezS81 ===
Productos encontrados en pagina 1: 56
Productos encontrados en pagina 2: 56
Productos encontrados en pagina 3: 56
Productos encontrados en pagina 4: 56
Productos encontrados en pagina 5: 56
Productos encontrados en pagina 6: 49
Timeout esperando productos en pagina 7
Productos encontrados en pagina 7: 0
Completados desde diccionario2: 293 de 266 sin color
Datos exportados a Liverpool_IOT_2026-10-23.xlsx


,Marca,Nombre,Color,Capacidad,CodigoProducto,SKU,URL,FechaConsulta,Precio_normal,Precio_descuento,Porcentaje_descuento,Categoria
0,OURA,Anillo inteligente Ring 5 unisex,ROSA CLARO,NA,1206254552,1206254609,https://www.liverpool.com.mx/tienda/pdp/anillo...,2026-10-23 10:17:19,"$8,499.00",No discount,No discount,Smartwatch
1,OURA,Anillo inteligente Ring 5 unisex,NEGRO,NA,1206254552,1206254609,https://www.liverpool.com.mx/tienda/pdp/anillo...,2026-10-23 10:17:19,"$8,499.00",No discount,No discount,Smartwatch
3,APPLE,Watch SE 3 GPS Sports Band,NEGRO,NA,1209801538,1209340671,https://www.liverpool.com.mx/tienda/pdp/watch-...,2026-10-23 10:17:19,"$5,499.00",No discount,No discount,Smartwatch
4,APPLE,Watch SE 3 GPS Sports Band,BLANCO,NA,1209801538,1209340671,https://www.liverpool.com.mx/tienda/pdp/watch-...,2026-10-23 10:17:19,"$5,499.00",No discount,No discount,Smartwatch
5,SAMSUNG,Smartwatch Fit 3 con GPS,PLATEADO,NA,1148752594,1148752601,https://www.liverpool.com.mx/tienda/pdp/smartw...,2026-10-23 10:17:19,$999.00,No discount,No discount,Smartwatch
6,SAMSUNG,Smartwatch Fit 3 con GPS,GRIS,NA,1148752594,1148752601,https://www.liverpool.com.mx/tienda/pdp/smartw...,2026-10-23 10:17:19,$999.00,No discount,No discount,Smartwatch
7,SAMSUNG,Smartwatch Fit 3 con GPS,ROSA CLARO,NA,1148752594,1148752601,https://www.liverpool.com.mx/tienda/pdp/smartw...,2026-10-23 10:17:19,$999.00,No discount,No discount,Smartwatch
8,APPLE,Watch Series 12 aluminio GPS,BEIGE,NA,1210805821,1209369881,https://www.liverpool.com.mx/tienda/pdp/watch-...,2026-10-23 10:17:19,"$8,999.00",No discount,No discount,Smartwatch
9,APPLE,Watch Series 12 aluminio GPS,NEGRO,NA,1210805821,1209369881,https://www.liverpool.com.mx/tienda/pdp/watch-...,2026-10-23 10:17:19,"$8,999.00",No discount,No discount,Smartwatch
10,APPLE,Watch Series 12 aluminio GPS,AZUL,NA,1210805821,1209369881,https://www.liverpool.com.mx/tienda/pdp/watch-...,2026-10-23 10:17:19,"$8,999.00",No discount,No discount,Smartwatch


# Extraer Tamaño/Capacidad desde URLs Liverpool CSV
Lee las URLs del archivo `urls liverpool.csv` y extrae el tamaño (capacidad) de cada celular visitando la página de detalle.

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ===================== CONFIGURACIÓN =====================

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

# ===================== LEER URLs DEL CSV =====================

csv_path = r"c:\Users\c50012218\Downloads\urls liverpool.csv"
df_urls = pd.read_csv(csv_path, header=None, names=["URL"])
urls = df_urls["URL"].dropna().str.strip().tolist()
print(f"Total de URLs a procesar: {len(urls)}")

# ===================== FUNCIÓN PARA EXTRAER TAMAÑO DE UNA URL =====================

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def leer_variantes_tamano(driver):
    """Hace clic en cada tamaño habilitado y regresa [(tamano, skuid)] leyendo el skuid de la URL."""
    soup = BeautifulSoup(driver.page_source, "html.parser")
    size_rg = soup.find("div", attrs={"data-testid": re.compile(r"^ml-radio-group-size-picker")})
    variantes = []
    if not size_rg:
        return variantes
    for inp in size_rg.find_all("input", attrs={"name": "size-picker"}):
        if inp.has_attr("disabled"):
            continue
        tamano = inp.get("value", "").strip()
        try:
            label = driver.find_element(
                By.XPATH,
                f'//div[starts-with(@data-testid,"ml-radio-group-size-picker")]//label[normalize-space()="{tamano}"]'
            )
            driver.execute_script("arguments[0].click();", label)
            time.sleep(2)
        except Exception:
            continue
        variantes.append((tamano, extraer_sku(driver.current_url)))
    return variantes

def extraer_tamano_url(url):
    """Visita la URL del producto y extrae todos los tamaños disponibles."""
    resultados = []
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'section[data-testid$="-configurator"]'))
        )
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Extraer nombre y marca
        nombre_tag = soup.find("h1")
        nombre = nombre_tag.get_text(strip=True) if nombre_tag else ""
        marca_tag = soup.find("a", attrs={"data-testid": re.compile(r"-brand-link$")})
        marca = marca_tag.get_text(strip=True) if marca_tag else ""

        # Extraer SKU de la URL
        sku = extraer_sku(url)
        codigo = extraer_codigo_producto(url)

        # Recorrer el selector de tamaños
        variantes = leer_variantes_tamano(driver)

        if variantes:
            for tamano_text, size_skuid in variantes:
                resultados.append({
                    "Marca": marca,
                    "Nombre": nombre,
                    "CodigoProducto": codigo,
                    "SKU": size_skuid if size_skuid else sku,
                    "Tamano": tamano_text,
                    "URL": url
                })
        else:
            # No hay selector de tamaños, buscar en el nombre del producto
            m = re.search(r'(\d+)\s*(GB|TB)\b', nombre, re.IGNORECASE)
            tamano = f"{m.group(1)} {m.group(2).upper()}" if m else ""
            resultados.append({
                "Marca": marca,
                "Nombre": nombre,
                "CodigoProducto": codigo,
                "SKU": sku,
                "Tamano": tamano,
                "URL": url
            })

        driver.quit()
    except Exception as e:
        print(f"  [ERROR] {url[:80]}... -> {e}")
        if driver:
            try:
                driver.quit()
            except:
                pass
        # Agregar entrada con tamaño vacío para no perder la URL
        resultados.append({
            "Marca": "",
            "Nombre": "",
            "CodigoProducto": "",
            "SKU": extraer_sku(url),
            "Tamano": "",
            "URL": url
        })
    return resultados

# ===================== PROCESAR EN PARALELO (BATCHES DE 2) =====================

BATCH_SIZE = 2
all_results = []
total = len(urls)

print(f"\n=== Procesando {total} URLs en paralelo (batches de {BATCH_SIZE}) ===\n")

for i in range(0, total, BATCH_SIZE):
    batch = urls[i:i+BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"Batch {batch_num}/{total_batches} ({i+1}-{min(i+BATCH_SIZE, total)} de {total})...")

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = {executor.submit(extraer_tamano_url, url): url for url in batch}
        for future in as_completed(futures):
            url = futures[future]
            try:
                resultado = future.result()
                all_results.extend(resultado)
            except Exception as e:
                print(f"  [ERROR] Batch error para {url[:60]}...: {e}")

    print(f"  Acumulado: {len(all_results)} registros")

# ===================== EXPORTAR RESULTADOS =====================

if all_results:
    df_tamanos = pd.DataFrame(all_results)
    df_tamanos = df_tamanos.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_tamanos_{fecha_hoy}.xlsx"
    df_tamanos.to_excel(nombre_archivo, index=False)

    print(f"\n{'='*60}")
    print(f"RESUMEN")
    print(f"{'='*60}")
    print(f"URLs procesadas: {total}")
    print(f"Registros totales (con variantes de tamaño): {len(df_tamanos)}")
    print(f"Con tamaño encontrado: {len(df_tamanos[df_tamanos['Tamano'] != ''])}")
    print(f"Sin tamaño: {len(df_tamanos[df_tamanos['Tamano'] == ''])}")
    print(f"Exportado a: {nombre_archivo}")
    print(f"\nPrimeros registros:")
    print(df_tamanos.head(10).to_string(index=False))
else:
    print("No se encontraron resultados.")

Total de URLs a procesar: 66

=== Procesando 66 URLs en paralelo (batches de 2) ===

Batch 1/33 (1-2 de 66)...
  Acumulado: 2 registros
Batch 2/33 (3-4 de 66)...
  Acumulado: 4 registros
Batch 3/33 (5-6 de 66)...
  Acumulado: 6 registros
Batch 4/33 (7-8 de 66)...
  Acumulado: 8 registros
Batch 5/33 (9-10 de 66)...
  Acumulado: 10 registros
Batch 6/33 (11-12 de 66)...
  Acumulado: 12 registros
Batch 7/33 (13-14 de 66)...
  Acumulado: 14 registros
Batch 8/33 (15-16 de 66)...
  Acumulado: 16 registros
Batch 9/33 (17-18 de 66)...
  Acumulado: 18 registros
Batch 10/33 (19-20 de 66)...
  Acumulado: 20 registros
Batch 11/33 (21-22 de 66)...
  [ERROR] https://www.liverpool.com.mx/tienda/pdp/samsung-galaxy-s10e-de-128gb-amoled-5.8-... -> Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6892f10b5+14cc5]
	chromedriver!GetHandleVerifier [0x7ff6892f1120+14d30]
	chromedriver!(No symbol) [0x7ff68905db6d]
	chromedriver!(No symbol) [0x7ff6890b7082]
	chromedriver!(No symbol) [0x7ff6890b738c]


# Scraping por Vendedores (sin Liverpool)
Extrae celulares de la categoría general cat5150024, iterando por cada filtro de vendedor (excepto Liverpool).
Agrega una columna "Vendedor" con el nombre del filtro seleccionado.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
import difflib
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

# ===================== FUNCIONES AUXILIARES =====================

COLORES_CONOCIDOS = [
    "NEGRO", "NEGRO CARBON", "NEGRO MEDIANOCHE", "NEGRO TITANIO", "NEGRO MATE",
    "BLANCO", "BLANCO ESTRELLA", "BLANCO PERLA", "BLANCO TITANIO",
    "GRIS", "GRIS TITANIO", "GRIS ESPACIAL", "GRIS PERLA", "GRIS GRAFITO", "GRIS PLATA",
    "PLATA", "PLATEADO", "PLATA TITANIO",
    "DORADO", "ORO", "DORADO TITANIO", "CHAMPAGNE",
    "AZUL", "AZUL CLARO", "AZUL OBSCURO", "AZUL OSCURO", "AZUL MARINO",
    "AZUL TITANIO", "AZUL NOCHE", "AZUL HIELO", "AZUL CIELO", "AZUL COBALTO",
    "AZUL REAL", "AZUL ZAFIRO", "AZUL PROFUNDO", "AZUL MEDIANOCHE", "AZUL OCEANO",
    "ROJO", "ROJO OSCURO", "ROJO CORAL", "ROJO TITANIO",
    "VERDE", "VERDE OLIVA", "VERDE MENTA", "VERDE SALVIA", "VERDE TITANIO",
    "VERDE CLARO", "VERDE OSCURO", "VERDE BOSQUE",
    "ROSA", "ROSA CLARO", "ROSA MELOCOTON", "ROSA PERLA", "ROSA TITANIO",
    "MORADO", "MORADO OSCURO", "VIOLETA", "VIOLET", "LILA", "LAVANDA", "PURPURA",
    "NARANJA", "SALMON", "CORAL",
    "AMARILLO", "CAFE", "MARRON", "BEIGE", "CREMA", "DESIERTO",
    "TURQUESA", "CYAN", "AQUA", "MENTA",
    "INDIGO", "ULTRAMARINO", "TITANIO", "NATURAL", "MEDIA NOCHE",
]

def completar_color(color_truncado):
    color_up = color_truncado.upper().strip()
    if not color_up:
        return ""
    if color_up in COLORES_CONOCIDOS:
        return color_up
    prefijo_matches = [c for c in COLORES_CONOCIDOS if c.startswith(color_up)]
    if len(prefijo_matches) == 1:
        return prefijo_matches[0]
    if len(prefijo_matches) > 1:
        return min(prefijo_matches, key=len)
    close = difflib.get_close_matches(color_up, COLORES_CONOCIDOS, n=1, cutoff=0.75)
    if close:
        return close[0]
    return ""

def limpiar_precio_con_rango(texto):
    if not texto:
        return texto
    if ' - ' in texto:
        return texto.split(' - ')[0].strip()
    partes = re.split(r'-(?=\$)', texto)
    if len(partes) >= 2 and partes[0].startswith('$'):
        return partes[0].strip()
    return texto

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    if '$' in str(precio_raw):
        cleaned = re.sub(r'\s+', ' ', precio_raw.strip())
        return limpiar_precio_con_rango(cleaned)
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori_str = precio_normal.split(' - ')[0] if ' - ' in str(precio_normal) else precio_normal
        p_desc_str = precio_descuento.split(' - ')[0] if ' - ' in str(precio_descuento) else precio_descuento
        p_ori = float(re.sub(r'[^\d.]', '', p_ori_str).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', p_desc_str).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def precios_son_rango(normal_text, desc_text):
    try:
        n = float(re.sub(r'[^\d.]', '', normal_text.split(' - ')[0].replace(',', '')))
        d = float(re.sub(r'[^\d.]', '', desc_text.split(' - ')[0].replace(',', '')))
        return n < d
    except Exception:
        return False

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def es_smartwatch(nombre):
    return "smartwatch" in (nombre or "").lower()

def extraer_colores_de_card(product):
    color_picker = product.find("div", attrs={"aria-label": "Color picker"})
    colores = []
    if color_picker:
        for btn in color_picker.find_all("button", attrs={"aria-label": "color"}):
            title = btn.get("title", "")
            if title:
                partes = [p.strip() for p in title.split(",")]
                color = completar_color(partes[-1]) if partes else ""
                capacidad = partes[-2] if len(partes) >= 3 else ""
                colores.append((color, capacidad))
    if not colores:
        colores = [("", "")]
    return colores

def extraer_precios_de_contenedor(element):
    price_container = element.find("div", attrs={"data-testid": re.compile(r"\d+-price$")})
    search_in = price_container if price_container else element
    precio_desc_tag = (
        search_in.find("span", attrs={"data-testid": "discounted"}) or
        search_in.find("span", class_="text-price-primary")
    )
    precio_normal_tag = (
        search_in.find("span", attrs={"data-testid": "original"}) or
        search_in.find("span", attrs={"data-testid": re.compile(r"-price-original-prices$")})
    )
    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
    return precio_normal_tag, precio_desc_tag, precio_normal, precio_desc

def extraer_detalle_con_vendedor(url, options):
    """
    Visita el detalle del producto una sola vez.
    Lee el vendedor desde a[data-testid$='-mkp-seller-link'] y extrae todas las variantes.
    Retorna (resultados, vendedor_nombre).
    resultados: lista de (color_name, codigo, sku_var, size_name, precio_normal, precio_desc)
    """
    resultados = []
    vendedor_det = ""
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)

        # Leer vendedor (no bloquear si no aparece)
        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid$="-mkp-seller-link"]'))
            )
        except Exception:
            pass
        soup_v = BeautifulSoup(driver.page_source, "html.parser")
        seller_link = soup_v.find("a", attrs={"data-testid": re.compile(r"-mkp-seller-link$")})
        if seller_link:
            vendedor_det = seller_link.get("aria-label", "") or seller_link.get_text(strip=True)

        # Esperar configurador
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'details[data-testid*="configurator"]'))
            )
        except Exception:
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-testid*="configurator-price"]'))
                )
            except Exception:
                pass
        time.sleep(1)

        def leer_estado_pagina():
            s = BeautifulSoup(driver.page_source, "html.parser")

            color_name = ""
            color_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-color-picker")})
            if color_det:
                summ = color_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2:
                        color_name = ps[1].text.strip()

            if not color_name:
                image_picker = s.find("div", attrs={"data-testid": "ml-image-picker"})
                if image_picker:
                    for btn in image_picker.find_all("button"):
                        check = btn.find("div", class_=lambda c: c and "bg-prim-500" in c)
                        if check:
                            p_tag = btn.find("p")
                            if p_tag:
                                color_name = p_tag.get_text(strip=True)
                            break

            if not color_name:
                page_text = s.get_text()
                m = re.search(r"Color:\s*([A-Za-z][A-Za-z\s]+?)(?:\s*keyboard|\s*Tama)", page_text)
                if m:
                    color_name = m.group(1).strip()

            size_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-size-picker")})
            size_name = ""
            if size_det:
                summ = size_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2 and "Seleccione" not in ps[1].text:
                        size_name = ps[1].text.strip()
            if not size_name:
                page_text = s.get_text()
                m = re.search(r"Tama.o:\s*(\d+\s*GB)", page_text, re.I)
                if m:
                    size_name = m.group(1).strip()

            price_container = s.find("div", attrs={"data-testid": re.compile(r"\d+-configurator-price$")})
            if price_container:
                precio_desc_tag = price_container.find("span", attrs={"data-testid": "discounted"})
                precio_normal_tag = price_container.find("span", attrs={"data-testid": "original"})
            else:
                precio_desc_tag = (
                    s.find("span", attrs={"data-testid": "discounted"}) or
                    s.find("span", class_="text-price-primary")
                )
                precio_normal_tag = s.find("span", attrs={"data-testid": "original"})
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            if not precio_desc and not precio_normal:
                price_span = s.find("span", class_=re.compile(r"text-heading-2xl"))
                if price_span:
                    precio_desc = price_span.get_text(strip=True)
            return color_name, size_name, precio_normal, precio_desc

        soup = BeautifulSoup(driver.page_source, "html.parser")

        color_picker_div = soup.find("div", attrs={"data-testid": "ml-color-picker"})
        color_testids = []
        if color_picker_div:
            for btn in color_picker_div.find_all("button", attrs={"aria-label": "color"}):
                tid = btn.get("data-testid", "")
                title = btn.get("title", "")
                color_hint = ""
                if title:
                    partes = [p.strip() for p in title.split(",")]
                    color_hint = completar_color(partes[-1]) if partes else ""
                if tid:
                    color_testids.append((tid, color_hint))

        if not color_testids:
            image_picker_div = soup.find("div", attrs={"data-testid": "ml-image-picker"})
            if image_picker_div:
                for btn in image_picker_div.find_all(
                    "button", attrs={"data-testid": re.compile(r"^ml-image-picker-image-\d+$")}
                ):
                    tid = btn.get("data-testid", "")
                    p_tag = btn.find("p")
                    color_hint = p_tag.get_text(strip=True) if p_tag else ""
                    if not color_hint:
                        img = btn.find("img")
                        if img:
                            color_hint = re.sub(r'\s+\d+$', '', img.get("alt", "")).strip()
                    if tid:
                        color_testids.append((tid, color_hint))

        size_rg = soup.find("div", attrs={"data-testid": re.compile(r"^ml-radio-group-size-picker")})
        size_texts = []
        if size_rg:
            for label in size_rg.find_all("label"):
                span = label.find("span")
                if span and span.text.strip():
                    size_texts.append(span.text.strip())

        combinaciones_unicas = set()

        if not color_testids and not size_texts:
            color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
            codigo = extraer_codigo_producto(driver.current_url)
            sku_var = extraer_sku(driver.current_url)
            clave = (color_name, size_name)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))
        else:
            if not color_testids:
                color_testids = [(None, "")]
            if not size_texts:
                size_texts = [None]

            for color_testid, color_hint in color_testids:
                if color_testid:
                    try:
                        btn = driver.find_element(By.CSS_SELECTOR, f'button[data-testid="{color_testid}"]')
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(1.5)
                    except Exception:
                        pass

                for size_text in size_texts:
                    if size_text:
                        try:
                            label = driver.find_element(
                                By.XPATH, f'//label[.//span[normalize-space()="{size_text}"]]'
                            )
                            driver.execute_script("arguments[0].click();", label)
                            time.sleep(1.5)
                        except Exception:
                            pass

                    color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
                    codigo = extraer_codigo_producto(driver.current_url)
                    sku_var = extraer_sku(driver.current_url)

                    if not color_name and color_hint:
                        color_name = color_hint
                    if not size_name and size_text:
                        size_name = size_text
                    if not precio_desc and not precio_normal:
                        continue

                    clave = (color_name, size_name)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))

    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    finally:
        if driver:
            driver.quit()
    return resultados, vendedor_det

def clasificar_productos_pagina(soup_page, vendedor_nombre, fecha_consulta):
    """
    Lee una página de listado. Los productos con precio rango/sin color van a la cola
    diferida. Los demás se retornan directamente con el vendedor del listado.
    Retorna (direct_data, nuevos_deferred).
    """
    direct_data = []
    nuevos_deferred = []

    products = soup_page.find_all("a", href=re.compile(r"^/tienda/pdp/"))

    for product in products:
        marca = product.find("h4")
        nombre = product.find("h3")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue

        href = product.get("href", "")
        url_cel = "https://www.liverpool.com.mx" + href if href.startswith("/tienda/pdp/") else ""

        precio_normal_tag, precio_desc_tag, precio_normal_text, precio_desc_text = extraer_precios_de_contenedor(product)
        colores = extraer_colores_de_card(product)
        todos_vacios = all(c == "" for c, _ in colores)

        es_solo_desc = (not precio_normal_tag or not precio_normal_text) and precio_desc_tag and precio_desc_text
        es_rango = (
            '-' in precio_normal_text or '-' in precio_desc_text or
            (precio_normal_text and precio_desc_text and precios_son_rango(precio_normal_text, precio_desc_text))
        )
        necesita_visita = url_cel and (es_rango or todos_vacios)

        base_row = {
            "Marca": marca.text.strip() if marca else "",
            "Nombre": nombre_texto,
            "FechaConsulta": fecha_consulta,
            "Vendedor": vendedor_nombre,
        }

        if necesita_visita:
            # Agregar a cola diferida (deduplicación se hace al añadir a deferred_queue)
            nuevos_deferred.append({
                "url": url_cel,
                "href": href,
                "Marca": base_row["Marca"],
                "Nombre": nombre_texto,
                "FechaConsulta": fecha_consulta,
            })
        elif es_solo_desc:
            for color_nombre, capacidad_nombre in colores:
                direct_data.append({**base_row,
                    "Color": color_nombre, "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "Precio_normal": limpiar_precio_con_rango(formatear_precio(precio_desc_text)),
                    "Precio_descuento": "No discount",
                })
        else:
            for color_nombre, capacidad_nombre in colores:
                direct_data.append({**base_row,
                    "Color": color_nombre, "Capacidad": capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal_tag else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc_tag else "No discount",
                })

    return direct_data, nuevos_deferred

# ===================== CONFIGURACIÓN =====================

BASE_URL = "https://www.liverpool.com.mx/tienda/celulares/cat5150024"

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data_vendedores = []   # productos directos (sin visita al detalle)
deferred_queue = []          # URLs a visitar en fase 2
deferred_urls_set = set()    # para deduplicación thread-safe
lock = threading.Lock()

# ===================== PASO 1: OBTENER FILTROS DE "Vendido por" =====================

print("=== Obteniendo filtros de 'Vendido por' ===")
driver = webdriver.Chrome(options=options)
driver.get(BASE_URL)
time.sleep(12)

# Expandir "Vendido por" si está colapsado
try:
    for btn in driver.find_elements(By.CSS_SELECTOR, 'button[data-testid="button-dropdown-filter"]'):
        if "Vendido por" in btn.text:
            arrow_down = btn.find_elements(By.CSS_SELECTOR, '[data-testid="keyboard-arrow-down-icon"]')
            if arrow_down:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(3)
                print("  Sección 'Vendido por' expandida.")
            else:
                print("  Sección 'Vendido por' ya expandida.")
            break
except Exception as e:
    print(f"  No se pudo interactuar con 'Vendido por': {e}")

# Clic en "Ver más" del filtro de vendedores
try:
    ver_mas_els = driver.find_elements(
        By.CSS_SELECTOR, 'button[data-testid$="seller-filter-sellers-checkbox-group-show-all-items-btn"]'
    )
    if ver_mas_els:
        driver.execute_script("arguments[0].click();", ver_mas_els[0])
        time.sleep(3)
        print("  'Ver más' clickeado.")
    else:
        print("  No se encontró 'Ver más' en 'Vendido por'.")
except Exception as e:
    print(f"  Error buscando 'Ver más': {e}")

time.sleep(2)
soup = BeautifulSoup(driver.page_source, "html.parser")
vendedores_info = []

vendido_btn_soup = None
for btn in soup.find_all("button", attrs={"data-testid": "button-dropdown-filter"}):
    if "Vendido por" in btn.get_text():
        vendido_btn_soup = btn
        break

if vendido_btn_soup:
    container = vendido_btn_soup.find_parent()
    while container and container.name not in ["body", "html"]:
        items = container.find_all("div", attrs={"data-testid": re.compile(r"^item-")})
        if items:
            print(f"  Encontrados {len(items)} items en 'Vendido por'")
            for item in items:
                testid = item.get("data-testid", "")
                span = item.find("span")
                vendor_name = span.get_text(strip=True) if span else testid.replace("item-", "")
                if vendor_name and "liverpool" not in vendor_name.lower():
                    vendedores_info.append({"nombre": vendor_name, "testid": testid})
                    print(f"    + {vendor_name} ({testid})")
            break
        container = container.find_parent()
else:
    print("  [DEBUG] No se encontró botón 'Vendido por'. Items en página:")
    for item in soup.find_all("div", attrs={"data-testid": re.compile(r"^item-")}):
        span = item.find("span")
        print(f"    {item.get('data-testid','')}: {span.get_text(strip=True) if span else ''}")

vistos = set()
vendedores_info = [v for v in vendedores_info if not (v["nombre"] in vistos or vistos.add(v["nombre"]))]

driver.quit()

print(f"\nVendedores encontrados (sin Liverpool): {len(vendedores_info)}")
for v in vendedores_info:
    print(f"  - {v['nombre']} ({v['testid']})")

# ===================== PASO 2: SCRAPING POR VENDEDOR (sin visitas al detalle) =====================

def procesar_vendedor(vendedor):
    vendedor_nombre = vendedor["nombre"]
    vendedor_testid = vendedor["testid"]
    resultados_directos = []

    print(f"\n{'='*60}")
    print(f"[INICIO] Procesando: {vendedor_nombre} ({vendedor_testid})")
    print(f"{'='*60}")

    page = 1
    max_pages = 20
    filter_url = None

    try:
        driver = webdriver.Chrome(options=options)
    except Exception as e:
        print(f"  [ERROR] No se pudo abrir Chrome para {vendedor_nombre}: {e}")
        return resultados_directos

    try:
        driver.get(BASE_URL)
        time.sleep(10)

        # Expandir "Vendido por" si colapsado
        try:
            for btn in driver.find_elements(By.CSS_SELECTOR, 'button[data-testid="button-dropdown-filter"]'):
                if "Vendido por" in btn.text:
                    arrow_down = btn.find_elements(By.CSS_SELECTOR, '[data-testid="keyboard-arrow-down-icon"]')
                    if arrow_down:
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(2)
                    break
        except Exception:
            pass

        # Clic en el item del vendedor
        try:
            item_el = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, f'div[data-testid="{vendedor_testid}"]'))
            )
            driver.execute_script("arguments[0].click();", item_el)
            time.sleep(8)
            filter_url = driver.current_url

            if filter_url == BASE_URL:
                parent_el = item_el.find_element(By.XPATH, "..")
                driver.execute_script("arguments[0].click();", parent_el)
                time.sleep(8)
                filter_url = driver.current_url
        except Exception as e:
            print(f"  [ERROR] No se pudo hacer clic en {vendedor_testid}: {e}")
            driver.quit()
            return resultados_directos

        print(f"  [{vendedor_nombre}] URL de filtro: {filter_url}")

        while page <= max_pages:
            if page > 1:
                if "/page-" not in filter_url:
                    page_url = filter_url + f"/page-{page}"
                else:
                    page_url = re.sub(r'/page-\d+', f'/page-{page}', filter_url)
                driver.get(page_url)
                time.sleep(10)

            for scroll_pos in range(800, 6000, 800):
                driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
                time.sleep(0.5)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            direct_data, nuevos_deferred = clasificar_productos_pagina(soup, vendedor_nombre, fecha_consulta)

            # Agregar diferidos a la cola global (deduplicando por URL)
            with lock:
                for item in nuevos_deferred:
                    if item["url"] not in deferred_urls_set:
                        deferred_urls_set.add(item["url"])
                        deferred_queue.append(item)

            resultados_directos.extend(direct_data)

            if not direct_data and not nuevos_deferred:
                print(f"  [{vendedor_nombre}] Página {page}: Sin productos. Fin.")
                break

            print(f"  [{vendedor_nombre}] Página {page}: {len(direct_data)} directos, {len(nuevos_deferred)} diferidos")

            pagination = soup.find("a", href=re.compile(f"page-{page+1}"))
            next_page = soup.find("a", string=re.compile(r"Siguiente|Next", re.IGNORECASE))
            next_button = soup.find("button", string=re.compile(r"Siguiente|Next", re.IGNORECASE))

            if not next_page and not next_button and not pagination:
                print(f"  [{vendedor_nombre}] No hay más páginas.")
                break

            page += 1

    except Exception as e:
        print(f"  [{vendedor_nombre}] Error inesperado: {e}")
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    print(f"  [{vendedor_nombre}] Total directos: {len(resultados_directos)}")
    return resultados_directos

BATCH_SIZE = 2
print(f"\n=== Procesando {len(vendedores_info)} vendedores en paralelo (batches de {BATCH_SIZE}) ===")

for i in range(0, len(vendedores_info), BATCH_SIZE):
    batch = vendedores_info[i:i+BATCH_SIZE]
    print(f"\n>>> Batch {i//BATCH_SIZE + 1}: {[v['nombre'] for v in batch]}")

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        resultados = list(executor.map(procesar_vendedor, batch))

    for resultado in resultados:
        final_data_vendedores.extend(resultado)

    print(f">>> Batch {i//BATCH_SIZE + 1} completado. Directos acumulados: {len(final_data_vendedores)}")

# ===================== PASO 3: PROCESAR COLA DIFERIDA =====================

print(f"\n=== Procesando {len(deferred_queue)} URLs diferidas (deduplicadas) ===")

def procesar_item_diferido(item):
    url = item["url"]
    href = item["href"]
    print(f"  Visitando: {url.split('/')[-1][:60]}")

    variantes, vendedor_det = extraer_detalle_con_vendedor(url, options)

    resultados_item = []
    if variantes:
        combinaciones_unicas = set()
        for color_det, codigo_det, sku_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
            clave = (color_det, tamano_det)
            if clave in combinaciones_unicas:
                continue
            combinaciones_unicas.add(clave)
            if not precio_normal_det and precio_desc_det:
                precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det))
                precio_desc_final = "No discount"
            else:
                precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_normal_det))
                precio_desc_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det)) if precio_desc_det else "No discount"
            sku_url = f"https://www.liverpool.com.mx{href}?skuid={sku_det}" if sku_det else url
            resultados_item.append({
                "Marca": item["Marca"],
                "Nombre": item["Nombre"],
                "Color": color_det,
                "Capacidad": tamano_det,
                "CodigoProducto": codigo_det,
                "SKU": sku_det,
                "URL": sku_url,
                "FechaConsulta": item["FechaConsulta"],
                "Precio_normal": precio_normal_final,
                "Precio_descuento": precio_desc_final,
                "Vendedor": vendedor_det,
            })
    else:
        resultados_item.append({
            "Marca": item["Marca"],
            "Nombre": item["Nombre"],
            "Color": "", "Capacidad": "",
            "CodigoProducto": extraer_codigo_producto(url),
            "SKU": extraer_sku(url),
            "URL": url,
            "FechaConsulta": item["FechaConsulta"],
            "Precio_normal": "",
            "Precio_descuento": "No discount",
            "Vendedor": vendedor_det,
        })

    return resultados_item

BATCH_SIZE_DEFER = 2
for i in range(0, len(deferred_queue), BATCH_SIZE_DEFER):
    batch = deferred_queue[i:i+BATCH_SIZE_DEFER]
    total_batches = (len(deferred_queue) + BATCH_SIZE_DEFER - 1) // BATCH_SIZE_DEFER
    print(f"\n>>> Batch diferido {i//BATCH_SIZE_DEFER + 1}/{total_batches}")

    with ThreadPoolExecutor(max_workers=BATCH_SIZE_DEFER) as executor:
        resultados_batch = list(executor.map(procesar_item_diferido, batch))

    for resultado in resultados_batch:
        final_data_vendedores.extend(resultado)

    print(f">>> Total acumulado: {len(final_data_vendedores)} productos")

# ===================== EXPORTAR RESULTADOS =====================

if final_data_vendedores:
    df_vend = pd.DataFrame(final_data_vendedores)

    try:
        diccionario = pd.read_csv("diccionario liverpool 2.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df_vend["SKU"] = df_vend["SKU"].astype(str).str.strip()
            df_vend = df_vend.merge(diccionario[["SKU", "Capacidad"]], on="SKU", how="left")
            df_vend["Capacidad"] = df_vend["Capacidad_y"].combine_first(df_vend["Capacidad_x"])
            df_vend.drop(columns=[col for col in df_vend.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró 'diccionario liverpool 2.csv'.")

    df_vend["Porcentaje_descuento"] = df_vend.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )

    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento", "Vendedor"
    ]
    df_vend = df_vend[[c for c in columnas_final if c in df_vend.columns]]
    df_vend = df_vend.drop_duplicates()

    urls_con_color = set(df_vend[df_vend["Color"] != ""]["URL"])
    mask = (df_vend["Color"] == "") & (df_vend["URL"].isin(urls_con_color))
    df_vend = df_vend[~mask]

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_vendedores_{fecha_hoy}.xlsx"
    df_vend.to_excel(nombre_archivo, index=False)

    print(f"\n{'='*60}")
    print(f"RESUMEN FINAL")
    print(f"{'='*60}")
    print(f"Total de productos: {len(df_vend)}")
    print(f"Vendedores: {df_vend['Vendedor'].nunique()}")
    for vend in df_vend['Vendedor'].value_counts().index:
        print(f"  - {vend}: {len(df_vend[df_vend['Vendedor']==vend])} productos")
    print(f"Datos exportados a {nombre_archivo}")
    display(df_vend.head(20))
else:
    print("No se encontraron productos de vendedores externos.")

# Scraping Reacondicionados Liverpool
Scrapea la sección de reacondicionados de Liverpool, extrayendo además quién vende cada producto.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
import difflib
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

COLORES_CONOCIDOS = [
    "NEGRO", "NEGRO CARBON", "NEGRO MEDIANOCHE", "NEGRO TITANIO", "NEGRO MATE",
    "BLANCO", "BLANCO ESTRELLA", "BLANCO PERLA", "BLANCO TITANIO",
    "GRIS", "GRIS TITANIO", "GRIS ESPACIAL", "GRIS PERLA", "GRIS GRAFITO", "GRIS PLATA",
    "PLATA", "PLATEADO", "PLATA TITANIO",
    "DORADO", "ORO", "DORADO TITANIO", "CHAMPAGNE",
    "AZUL", "AZUL CLARO", "AZUL OBSCURO", "AZUL OSCURO", "AZUL MARINO",
    "AZUL TITANIO", "AZUL NOCHE", "AZUL HIELO", "AZUL CIELO", "AZUL COBALTO",
    "AZUL REAL", "AZUL ZAFIRO", "AZUL PROFUNDO", "AZUL MEDIANOCHE", "AZUL OCEANO",
    "ROJO", "ROJO OSCURO", "ROJO CORAL", "ROJO TITANIO",
    "VERDE", "VERDE OLIVA", "VERDE MENTA", "VERDE SALVIA", "VERDE TITANIO",
    "VERDE CLARO", "VERDE OSCURO", "VERDE BOSQUE",
    "ROSA", "ROSA CLARO", "ROSA MELOCOTON", "ROSA PERLA", "ROSA TITANIO",
    "MORADO", "MORADO OSCURO", "VIOLETA", "VIOLET", "LILA", "LAVANDA", "PURPURA",
    "NARANJA", "SALMON", "CORAL",
    "AMARILLO", "CAFE", "MARRON", "BEIGE", "CREMA", "DESIERTO",
    "TURQUESA", "CYAN", "AQUA", "MENTA",
    "INDIGO", "ULTRAMARINO", "TITANIO", "NATURAL", "MEDIA NOCHE",
]

def completar_color(color_truncado):
    color_up = color_truncado.upper().strip()
    if not color_up:
        return ""
    if color_up in COLORES_CONOCIDOS:
        return color_up
    prefijo_matches = [c for c in COLORES_CONOCIDOS if c.startswith(color_up)]
    if len(prefijo_matches) == 1:
        return prefijo_matches[0]
    if len(prefijo_matches) > 1:
        return min(prefijo_matches, key=len)
    close = difflib.get_close_matches(color_up, COLORES_CONOCIDOS, n=1, cutoff=0.75)
    if close:
        return close[0]
    return ""

def limpiar_precio_con_rango(texto):
    if not texto:
        return texto
    if ' - ' in texto:
        return texto.split(' - ')[0].strip()
    partes = re.split(r'-(?=\$)', texto)
    if len(partes) >= 2 and partes[0].startswith('$'):
        return partes[0].strip()
    return texto

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    if '$' in str(precio_raw):
        cleaned = re.sub(r'\s+', ' ', precio_raw.strip())
        return limpiar_precio_con_rango(cleaned)
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori_str = precio_normal.split(' - ')[0] if ' - ' in str(precio_normal) else precio_normal
        p_desc_str = precio_descuento.split(' - ')[0] if ' - ' in str(precio_descuento) else precio_descuento
        p_ori = float(re.sub(r'[^\d.]', '', p_ori_str).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', p_desc_str).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_capacidad_nombre(nombre):
    """Extrae la capacidad de almacenamiento del nombre del producto."""
    if not nombre:
        return ""
    # Patrones comunes: "128 GB", "256GB", "1 TB", "128gb", "512 gb"
    match = re.search(r'(\d+)\s*(GB|TB|gb|tb|Gb|Tb)', nombre)
    if match:
        return f"{match.group(1)} {match.group(2).upper()}"
    return ""

def es_smartwatch(nombre):
    return "smartwatch" in (nombre or "").lower()

def extraer_colores_de_card(product):
    color_picker = product.find("div", attrs={"aria-label": "Color picker"})
    colores = []
    if color_picker:
        for btn in color_picker.find_all("button", attrs={"aria-label": "color"}):
            title = btn.get("title", "")
            if title:
                partes = [p.strip() for p in title.split(",")]
                color = completar_color(partes[-1]) if partes else ""
                capacidad = partes[-2] if len(partes) >= 3 else ""
                colores.append((color, capacidad))
    if not colores:
        colores = [("", "")]
    return colores

def extraer_precios_de_contenedor(element):
    price_container = element.find("div", attrs={"data-testid": re.compile(r"\d+-price$")})
    search_in = price_container if price_container else element
    precio_desc_tag = (
        search_in.find("span", attrs={"data-testid": "discounted"}) or
        search_in.find("span", class_="text-price-primary")
    )
    precio_normal_tag = (
        search_in.find("span", attrs={"data-testid": "original"}) or
        search_in.find("span", attrs={"data-testid": re.compile(r"-price-original-prices$")})
    )
    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
    return precio_normal_tag, precio_desc_tag, precio_normal, precio_desc

def cargar_pagina_con_retry(driver, url, max_intentos=4, espera=10):
    for intento in range(max_intentos):
        try:
            driver.get(url)
            time.sleep(espera + random.uniform(2, 5))
            src = driver.page_source.lower()
            if "upstream connect error" in src or "connection termination" in src or "reset reason" in src:
                wait = 20 * (intento + 1) + random.uniform(5, 15)
                print(f"  [RETRY {intento+1}] Error de conexión, reintentando en {wait:.0f}s...")
                time.sleep(wait)
                continue
            return True
        except Exception as e:
            wait = 20 * (intento + 1)
            print(f"  [RETRY {intento+1}] Excepción: {e}, esperando {wait}s...")
            time.sleep(wait)
    print(f"  [FAIL] No se pudo cargar: {url}")
    return False

def extraer_detalle_con_vendedor(url, options):
    """
    Visita el detalle del producto una sola vez.
    Lee el vendedor desde a[data-testid$='-mkp-seller-link'] y extrae todas las variantes.
    Retorna (resultados, vendedor_nombre).
    resultados: lista de (color_name, codigo, sku_var, size_name, precio_normal, precio_desc)
    """
    resultados = []
    vendedor_det = ""
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        if not cargar_pagina_con_retry(driver, url, espera=8):
            return resultados, vendedor_det

        # Leer vendedor (no bloquear si no aparece)
        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid$="-mkp-seller-link"]'))
            )
        except Exception:
            pass
        soup_v = BeautifulSoup(driver.page_source, "html.parser")
        seller_link = soup_v.find("a", attrs={"data-testid": re.compile(r"-mkp-seller-link$")})
        if seller_link:
            vendedor_det = seller_link.get("aria-label", "") or seller_link.get_text(strip=True)

        # Esperar configurador
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'details[data-testid*="configurator"]'))
            )
        except Exception:
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-testid*="configurator-price"]'))
                )
            except Exception:
                pass
        time.sleep(1)

        def leer_estado_pagina():
            s = BeautifulSoup(driver.page_source, "html.parser")

            color_name = ""
            color_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-color-picker")})
            if color_det:
                summ = color_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2:
                        color_name = ps[1].text.strip()

            if not color_name:
                image_picker = s.find("div", attrs={"data-testid": "ml-image-picker"})
                if image_picker:
                    for btn in image_picker.find_all("button"):
                        check = btn.find("div", class_=lambda c: c and "bg-prim-500" in c)
                        if check:
                            p_tag = btn.find("p")
                            if p_tag:
                                color_name = p_tag.get_text(strip=True)
                            break

            if not color_name:
                page_text = s.get_text()
                m = re.search(r"Color:\s*([A-Za-z][A-Za-z\s]+?)(?:\s*keyboard|\s*Tama)", page_text)
                if m:
                    color_name = m.group(1).strip()

            size_det = s.find("details", attrs={"data-testid": re.compile(r"configurator-size-picker")})
            size_name = ""
            if size_det:
                summ = size_det.find("summary")
                if summ:
                    ps = summ.find_all("p")
                    if len(ps) >= 2 and "Seleccione" not in ps[1].text:
                        size_name = ps[1].text.strip()
            if not size_name:
                page_text = s.get_text()
                m = re.search(r"Tama.o:\s*(\d+\s*(?:GB|TB))", page_text, re.I)
                if m:
                    size_name = m.group(1).strip()

            price_container = s.find("div", attrs={"data-testid": re.compile(r"\d+-configurator-price$")})
            if price_container:
                precio_desc_tag = price_container.find("span", attrs={"data-testid": "discounted"})
                precio_normal_tag = price_container.find("span", attrs={"data-testid": "original"})
            else:
                precio_desc_tag = (
                    s.find("span", attrs={"data-testid": "discounted"}) or
                    s.find("span", class_="text-price-primary")
                )
                precio_normal_tag = s.find("span", attrs={"data-testid": "original"})
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            if not precio_desc and not precio_normal:
                price_span = s.find("span", class_=re.compile(r"text-heading-2xl"))
                if price_span:
                    precio_desc = price_span.get_text(strip=True)
            return color_name, size_name, precio_normal, precio_desc

        soup = BeautifulSoup(driver.page_source, "html.parser")

        color_picker_div = soup.find("div", attrs={"data-testid": "ml-color-picker"})
        color_testids = []
        if color_picker_div:
            for btn in color_picker_div.find_all("button", attrs={"aria-label": "color"}):
                tid = btn.get("data-testid", "")
                title = btn.get("title", "")
                color_hint = ""
                if title:
                    partes = [p.strip() for p in title.split(",")]
                    color_hint = completar_color(partes[-1]) if partes else ""
                if tid:
                    color_testids.append((tid, color_hint))

        if not color_testids:
            image_picker_div = soup.find("div", attrs={"data-testid": "ml-image-picker"})
            if image_picker_div:
                for btn in image_picker_div.find_all(
                    "button", attrs={"data-testid": re.compile(r"^ml-image-picker-image-\d+$")}
                ):
                    tid = btn.get("data-testid", "")
                    p_tag = btn.find("p")
                    color_hint = p_tag.get_text(strip=True) if p_tag else ""
                    if not color_hint:
                        img = btn.find("img")
                        if img:
                            color_hint = re.sub(r'\s+\d+$', '', img.get("alt", "")).strip()
                    if tid:
                        color_testids.append((tid, color_hint))

        size_rg = soup.find("div", attrs={"data-testid": re.compile(r"^ml-radio-group-size-picker")})
        size_texts = []
        if size_rg:
            for label in size_rg.find_all("label"):
                span = label.find("span")
                if span and span.text.strip():
                    size_texts.append(span.text.strip())

        combinaciones_unicas = set()

        if not color_testids and not size_texts:
            color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
            codigo = extraer_codigo_producto(driver.current_url)
            sku_var = extraer_sku(driver.current_url)
            clave = (color_name, size_name)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))
        else:
            if not color_testids:
                color_testids = [(None, "")]
            if not size_texts:
                size_texts = [None]

            for color_testid, color_hint in color_testids:
                if color_testid:
                    try:
                        btn = driver.find_element(By.CSS_SELECTOR, f'button[data-testid="{color_testid}"]')
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(1.5)
                    except Exception:
                        pass

                for size_text in size_texts:
                    if size_text:
                        try:
                            label = driver.find_element(
                                By.XPATH, f'//label[.//span[normalize-space()="{size_text}"]]'
                            )
                            driver.execute_script("arguments[0].click();", label)
                            time.sleep(1.5)
                        except Exception:
                            pass

                    color_name, size_name, precio_normal, precio_desc = leer_estado_pagina()
                    codigo = extraer_codigo_producto(driver.current_url)
                    sku_var = extraer_sku(driver.current_url)

                    if not color_name and color_hint:
                        color_name = color_hint
                    if not size_name and size_text:
                        size_name = size_text
                    if not precio_desc and not precio_normal:
                        continue

                    clave = (color_name, size_name)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color_name, codigo, sku_var, size_name, precio_normal, precio_desc))

    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    finally:
        if driver:
            driver.quit()
    return resultados, vendedor_det

# ============================================================
# CONFIGURACIÓN
# ============================================================
BASE_URL = "https://www.liverpool.com.mx/tienda/reacondicionados/catst23690313"

options = Options()
# options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data_reac = []
visited_urls_reac = set()

# ============================================================
# FUNCIÓN PARA SCRAPEAR UNA PÁGINA (usada para todas las páginas)
# El vendedor ya no aparece en la card del listado: se entra al detalle de cada producto.
# ============================================================
def scrape_page_reac(page):
    driver = webdriver.Chrome(options=options)
    # Página 1 usa la URL base, las demás usan /page-X
    url = BASE_URL if page == 1 else f"{BASE_URL}/page-{page}"
    if not cargar_pagina_con_retry(driver, url, espera=12):
        print(f"No se pudo cargar la página {page}.")
        driver.quit()
        return []
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/tienda/pdp/"]'))
        )
    except Exception:
        print(f"Timeout esperando productos en pagina {page}")
    for scroll_pos in range(800, 6000, 800):
        driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    products = soup.find_all("a", href=re.compile(r"^/tienda/pdp/"))
    print(f"Página {page} - Productos encontrados: {len(products)}")
    page_data = []
    for product in products:
        marca = product.find("h4")
        nombre = product.find("h3")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        href = product.get("href", "")
        url_cel = "https://www.liverpool.com.mx" + href
        if url_cel in visited_urls_reac:
            continue
        visited_urls_reac.add(url_cel)

        precio_normal_tag, precio_desc_tag, precio_normal_text, precio_desc_text = extraer_precios_de_contenedor(product)
        colores = extraer_colores_de_card(product)
        capacidad_nombre = extraer_capacidad_nombre(nombre_texto)

        print(f"  Entrando a detalle para vendedor+variantes: {nombre_texto[:50]}...")
        time.sleep(random.uniform(3, 6))
        variantes, vendedor = extraer_detalle_con_vendedor(url_cel, options)
        if not vendedor:
            vendedor = "Liverpool"

        if variantes:
            for color_det, codigo_det, sku_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                if not precio_normal_det and precio_desc_det:
                    precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det))
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = limpiar_precio_con_rango(formatear_precio(precio_normal_det))
                    precio_desc_final = limpiar_precio_con_rango(formatear_precio(precio_desc_det)) if precio_desc_det else "No discount"
                sku_url = f"https://www.liverpool.com.mx{href.split('?')[0]}?skuid={sku_det}" if sku_det else url_cel
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_det,
                    # Si el detalle no trajo capacidad, usar la extraída del nombre
                    "Capacidad": tamano_det if tamano_det else capacidad_nombre,
                    "CodigoProducto": codigo_det,
                    "SKU": sku_det,
                    "URL": sku_url,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final,
                    "Vendedor": vendedor
                })
        else:
            # Sin variantes en el detalle: usar los datos de la card
            es_solo_desc = (not precio_normal_tag or not precio_normal_text) and precio_desc_tag and precio_desc_text
            for color_nombre, capacidad_card in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre_texto,
                    "Color": color_nombre,
                    "Capacidad": capacidad_card if capacidad_card else capacidad_nombre,
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": limpiar_precio_con_rango(formatear_precio(precio_desc_text if es_solo_desc else precio_normal_text)),
                    "Precio_descuento": "No discount" if es_solo_desc or not precio_desc_text else limpiar_precio_con_rango(formatear_precio(precio_desc_text)),
                    "Vendedor": vendedor
                })
    return page_data

# Procesar páginas en lotes de 2 para evitar MemoryError
lotes = [[1, 2], [3, 4]]
for lote in lotes:
    print(f"\n--- Procesando lote: páginas {lote} ---")
    with ThreadPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(scrape_page_reac, lote))
    for page_data in results:
        final_data_reac.extend(page_data)
    print(f"--- Lote páginas {lote} completado. Total acumulado: {len(final_data_reac)} ---")

# ============================================================
# PROCESAR Y EXPORTAR
# ============================================================
if final_data_reac:
    df_reac = pd.DataFrame(final_data_reac)

    # Intentar usar diccionario de capacidades
    try:
        diccionario = pd.read_csv("diccionario liverpool 2.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df_reac["SKU"] = df_reac["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df_reac = df_reac.merge(diccionario, on="SKU", how="left")
            df_reac["Capacidad"] = df_reac["Capacidad_y"].combine_first(df_reac["Capacidad_x"])
            df_reac.drop(columns=[col for col in df_reac.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró 'diccionario liverpool 2.csv'. Se continúa sin enriquecer capacidad.")

    # Porcentaje de descuento
    df_reac["Porcentaje_descuento"] = df_reac.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )

    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento", "Vendedor"
    ]
    df_reac = df_reac[columnas_final]
    df_reac = df_reac.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_reacondicionados_{fecha_hoy}.xlsx"
    df_reac.to_excel(nombre_archivo, index=False)
    print(f"\nResumen Reacondicionados:")
    print(f"Total de productos: {len(df_reac)}")
    print(f"Vendedores encontrados: {df_reac['Vendedor'].unique().tolist()}")
    print(f"Datos exportados a {nombre_archivo}")
else:
    print("No se encontraron productos reacondicionados.")


--- Procesando lote: páginas [1, 2] ---
Página 2 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: Redmi 9C IPS 6.5 pulgadas Desbloqueado Reacondicio...
Página 2 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: Redmi 9C IPS 6.5 pulgadas Desbloqueado Reacondicio...
Página 1 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: iPhone 14 Pro Max 6.7 pulgadas AMOLED Reacondicion...
Página 1 - Productos encontrados: 56
  Entrando a detalle para vendedor+capacidad: iPhone 14 Pro Max 6.7 pulgadas AMOLED Reacondicion...
  Entrando a detalle para vendedor+capacidad: Hyline Pro LCD IPS 6.6 pulgadas Reacondicionado...
  Entrando a detalle para vendedor+capacidad: Hyline Pro LCD IPS 6.6 pulgadas Reacondicionado...
  Entrando a detalle para vendedor+capacidad: Galaxy S24 Fe Dynamic AMOLED 2X 6.7 pulgadas Reaco...
  Entrando a detalle para vendedor+capacidad: Galaxy S24 Fe Dynamic AMOLED 2X 6.7 pulgadas Reaco...
  Entrando a deta

# Manual

In [1]:
# Versión modificada: No scrappear cuando hay "-" en el precio, escribir a mano en su lugar
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
import difflib
from datetime import datetime

COLORES_CONOCIDOS = [
    "NEGRO", "NEGRO CARBON", "NEGRO MEDIANOCHE", "NEGRO TITANIO", "NEGRO MATE",
    "BLANCO", "BLANCO ESTRELLA", "BLANCO PERLA", "BLANCO TITANIO",
    "GRIS", "GRIS TITANIO", "GRIS ESPACIAL", "GRIS PERLA", "GRIS GRAFITO", "GRIS PLATA",
    "PLATA", "PLATEADO", "PLATA TITANIO",
    "DORADO", "ORO", "DORADO TITANIO", "CHAMPAGNE",
    "AZUL", "AZUL CLARO", "AZUL OBSCURO", "AZUL OSCURO", "AZUL MARINO",
    "AZUL TITANIO", "AZUL NOCHE", "AZUL HIELO", "AZUL CIELO", "AZUL COBALTO",
    "AZUL REAL", "AZUL ZAFIRO", "AZUL PROFUNDO", "AZUL MEDIANOCHE", "AZUL OCEANO",
    "ROJO", "ROJO OSCURO", "ROJO CORAL", "ROJO TITANIO",
    "VERDE", "VERDE OLIVA", "VERDE MENTA", "VERDE SALVIA", "VERDE TITANIO",
    "VERDE CLARO", "VERDE OSCURO", "VERDE BOSQUE",
    "ROSA", "ROSA CLARO", "ROSA MELOCOTON", "ROSA PERLA", "ROSA TITANIO",
    "MORADO", "MORADO OSCURO", "VIOLETA", "VIOLET", "LILA", "LAVANDA", "PURPURA",
    "NARANJA", "SALMON", "CORAL",
    "AMARILLO", "CAFE", "MARRON", "BEIGE", "CREMA", "DESIERTO",
    "TURQUESA", "CYAN", "AQUA", "MENTA",
    "INDIGO", "ULTRAMARINO", "TITANIO", "NATURAL", "MEDIA NOCHE",
]

def completar_color(color_truncado):
    color_up = color_truncado.upper().strip()
    if not color_up:
        return ""
    if color_up in COLORES_CONOCIDOS:
        return color_up
    prefijo_matches = [c for c in COLORES_CONOCIDOS if c.startswith(color_up)]
    if len(prefijo_matches) == 1:
        return prefijo_matches[0]
    if len(prefijo_matches) > 1:
        return min(prefijo_matches, key=len)
    close = difflib.get_close_matches(color_up, COLORES_CONOCIDOS, n=1, cutoff=0.75)
    if close:
        return close[0]
    return ""

def limpiar_precio_con_rango(texto):
    if not texto:
        return texto
    if ' - ' in texto:
        return texto.split(' - ')[0].strip()
    partes = re.split(r'-(?=\$)', texto)
    if len(partes) >= 2 and partes[0].startswith('$'):
        return partes[0].strip()
    return texto

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    if '$' in str(precio_raw):
        cleaned = re.sub(r'\s+', ' ', precio_raw.strip())
        return limpiar_precio_con_rango(cleaned)
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    if match:
        return match.group(1)
    match = re.search(r'/(\d+)$', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def es_smartwatch(nombre):
    return "smartwatch" in (nombre or "").lower()

def extraer_colores_de_card(product):
    color_picker = product.find("div", attrs={"aria-label": "Color picker"})
    colores = []
    if color_picker:
        for btn in color_picker.find_all("button", attrs={"aria-label": "color"}):
            title = btn.get("title", "")
            if title:
                partes = [p.strip() for p in title.split(",")]
                color = completar_color(partes[-1]) if partes else ""
                capacidad = partes[-2] if len(partes) >= 3 else ""
                colores.append((color, capacidad))
    if not colores:
        colores = [("", "")]
    return colores

def extraer_precios_de_contenedor(element):
    price_container = element.find("div", attrs={"data-testid": re.compile(r"\d+-price$")})
    search_in = price_container if price_container else element
    precio_desc_tag = (
        search_in.find("span", attrs={"data-testid": "discounted"}) or
        search_in.find("span", class_="text-price-primary")
    )
    precio_normal_tag = (
        search_in.find("span", attrs={"data-testid": "original"}) or
        search_in.find("span", attrs={"data-testid": re.compile(r"-price-original-prices$")})
    )
    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
    return precio_normal_tag, precio_desc_tag, precio_normal, precio_desc

def cargar_pagina_con_retry(driver, url, max_intentos=4, espera=10):
    for intento in range(max_intentos):
        try:
            driver.get(url)
            time.sleep(espera + random.uniform(2, 5))
            src = driver.page_source.lower()
            if "upstream connect error" in src or "connection termination" in src or "reset reason" in src:
                wait = 20 * (intento + 1) + random.uniform(5, 15)
                print(f"  [RETRY {intento+1}] Error de conexión, reintentando en {wait:.0f}s...")
                time.sleep(wait)
                continue
            return True
        except Exception as e:
            wait = 20 * (intento + 1)
            print(f"  [RETRY {intento+1}] Excepción: {e}, esperando {wait}s...")
            time.sleep(wait)
    print(f"  [FAIL] No se pudo cargar: {url}")
    return False

options = Options()
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

BASE_URL = "https://www.liverpool.com.mx/tienda/N-hn2yreqK8COZekZ8ajJySwGcgqAL7Wnt_UndKuRj95BZQCFVNND4Qg-QZJwWijHi"

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []

def scrape_page_modified(page):
    """Scrapea una página del listado sin entrar al detalle de los productos"""
    driver = webdriver.Chrome(options=options)
    url = BASE_URL if page == 1 else f"{BASE_URL}/page-{page}"
    if not cargar_pagina_con_retry(driver, url, espera=12):
        print(f"No se pudo cargar la página {page}.")
        driver.quit()
        return []
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/tienda/pdp/"]'))
        )
    except Exception:
        print(f"Timeout esperando productos en pagina {page}")
    for scroll_pos in range(800, 6000, 800):
        driver.execute_script(f"window.scrollTo(0, {scroll_pos});")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    products = soup.find_all("a", href=re.compile(r"^/tienda/pdp/"))
    print(f"Página {page} - Productos encontrados: {len(products)}")

    page_data = []
    for product in products:
        marca = product.find("h4")
        nombre = product.find("h3")
        nombre_texto = nombre.text.strip() if nombre else ""

        if es_smartwatch(nombre_texto):
            continue

        url_cel = "https://www.liverpool.com.mx" + product.get("href", "")
        precio_normal_tag, precio_desc_tag, precio_normal_text, precio_desc_text = extraer_precios_de_contenedor(product)
        colores = extraer_colores_de_card(product)

        base_row = {
            "Marca": marca.text.strip() if marca else "",
            "Nombre": nombre_texto,
            "CodigoProducto": extraer_codigo_producto(url_cel),
            "SKU": extraer_sku(url_cel),
            "URL": url_cel,
            "FechaConsulta": fecha_consulta,
        }

        # MODIFICACIÓN: Si hay "-" en el precio, escribir "Manual Entry" en lugar de scrappear
        if '-' in precio_normal_text or '-' in precio_desc_text:
            for color_nombre, capacidad_nombre in colores:
                page_data.append({**base_row,
                    "Color": color_nombre,
                    "Capacidad": "Manual Entry",
                    "Precio_normal": "Manual Entry",
                    "Precio_descuento": "Manual Entry"
                })
            print(f"Página {page} - Producto con precio variable: {nombre_texto} - Se requiere entrada manual")
        # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
        elif (not precio_normal_tag or not precio_normal_text) and precio_desc_tag and precio_desc_text:
            for color_nombre, capacidad_nombre in colores:
                page_data.append({**base_row,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        else:
            for color_nombre, capacidad_nombre in colores:
                page_data.append({**base_row,
                    "Color": color_nombre,
                    "Capacidad": capacidad_nombre,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal_tag else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc_tag else "No discount"
                })

    return page_data

for page in range(1, 5):
    final_data.extend(scrape_page_modified(page))

# Procesar los datos finales
if final_data:
    df = pd.DataFrame(final_data)

    # Intentar cargar el diccionario para capacidades
    try:
        diccionario = pd.read_csv("diccionario liverpool.csv", encoding="utf-8")
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)

        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip()
            df["SKU"] = df["SKU"].astype(str).str.strip()
            diccionario = diccionario[["SKU", "Capacidad"]]
            df = df.merge(diccionario, on="SKU", how="left")

            # Solo actualizar capacidad si no es "Manual Entry"
            df["Capacidad"] = df.apply(
                lambda row: row["Capacidad_y"] if pd.notna(row["Capacidad_y"]) and row["Capacidad_x"] != "Manual Entry"
                else row["Capacidad_x"], axis=1
            )
            df.drop(columns=[col for col in df.columns if col.startswith("Capacidad_")], inplace=True)
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
    except FileNotFoundError:
        print("Advertencia: No se encontró el archivo 'diccionario liverpool.csv'.")

    # Calcular porcentaje de descuento solo para precios válidos
    def calcular_porcentaje_seguro(row):
        if row["Precio_normal"] == "Manual Entry" or row["Precio_descuento"] == "Manual Entry":
            return "Manual Entry"
        else:
            return calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"])

    df["Porcentaje_descuento"] = df.apply(calcular_porcentaje_seguro, axis=1)

    # Organizar columnas finales
    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
    ]
    df = df[columnas_final]

    # Eliminar duplicados
    df = df.drop_duplicates()

    # Contar productos que requieren entrada manual
    productos_manuales = df[df["Precio_normal"] == "Manual Entry"].shape[0]
    print(f"\nResumen:")
    print(f"Total de productos procesados: {len(df)}")
    print(f"Productos que requieren entrada manual de precios: {productos_manuales}")

    # Guardar archivo
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"liverpool_colores_{fecha_hoy}_sin_scraping.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")

    # Mostrar algunos productos que requieren entrada manual
    if productos_manuales > 0:
        print(f"\nEjemplos de productos que requieren entrada manual:")
        manual_products = df[df["Precio_normal"] == "Manual Entry"][["Marca", "Nombre", "URL"]].head(5)
        for idx, row in manual_products.iterrows():
            print(f"- {row['Marca']} {row['Nombre']}")
else:
    print("No se encontraron productos. Ajusta el selector de productos si es necesario.")

Productos encontrados: 56
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: iPhone 16 6.1 pulgadas Super Retina XDR - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy A16 Super AMOLED 6.7 pulgadas - Se requiere entrada manual
Producto con precio variable encontrado: Galaxy S25 Ultra Dynamic AMOLED 2X 6.9 pulgadas - Se requiere 

In [11]:
import pandas as pd
import re
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

def extraer_capacidad_de_nombre(nombre):
    if not nombre:
        return ""
    match = re.search(r"\b(\d+)\s*(TB|GB|MB)\b", str(nombre), re.IGNORECASE)
    if match:
        return f"{match.group(1)} {match.group(2).upper()}"
    return ""

def extraer_codigo_producto(url):
    match = re.search(r'/(\d+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    match = re.search(r'skuid=(\d+)', url)
    return match.group(1) if match else ""

def extraer_capacidad_de_url(url, driver):
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'section[data-testid$="-configurator"]'))
        )
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        sku = extraer_sku(url)

        # Botones de tamano (para celulares, tamano = capacidad de almacenamiento).
        # La pagina no preselecciona el tamano del skuid: se da clic en cada uno y se compara el skuid resultante.
        size_rg = soup.find("div", attrs={"data-testid": re.compile(r"^ml-radio-group-size-picker")})
        tamanos = []
        if size_rg:
            tamanos = [inp.get("value", "").strip() for inp in size_rg.find_all("input", attrs={"name": "size-picker"})
                       if not inp.has_attr("disabled")]
        if len(tamanos) == 1:
            return tamanos[0]
        for tamano in tamanos:
            try:
                label = driver.find_element(
                    By.XPATH,
                    f'//div[starts-with(@data-testid,"ml-radio-group-size-picker")]//label[normalize-space()="{tamano}"]'
                )
                driver.execute_script("arguments[0].click();", label)
                time.sleep(2)
            except Exception:
                continue
            if extraer_sku(driver.current_url) == sku:
                return tamano

        # Fallback: extraer del titulo del producto en la pagina
        nombre_tag = soup.find("h1")
        if nombre_tag:
            cap = extraer_capacidad_de_nombre(nombre_tag.get_text())
            if cap:
                return cap

    except Exception as e:
        print(f"  Error: {e}")
    return ""

# --- Cargar URLs desde 'urls liverpool.csv' ---
archivo_urls = "urls liverpool.csv"
if not os.path.exists(archivo_urls):
    print(f"No se encontro '{archivo_urls}'. Asegurate de que el archivo este en el directorio de trabajo.")
else:
    with open(archivo_urls, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    print(f"URLs cargadas: {len(urls)}")

    options_cap = Options()
    options_cap.add_argument("--headless")
    options_cap.add_argument("--disable-gpu")
    options_cap.add_argument("--window-size=1920,1080")
    options_cap.add_argument("--disable-blink-features=AutomationControlled")
    options_cap.add_experimental_option("excludeSwitches", ["enable-automation"])
    options_cap.add_experimental_option("useAutomationExtension", False)
    options_cap.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    resultados = []
    driver = webdriver.Chrome(options=options_cap)

    try:
        for i, url in enumerate(urls, 1):
            codigo = extraer_codigo_producto(url)
            sku = extraer_sku(url)
            print(f"[{i}/{len(urls)}] SKU: {sku} | {url[:70]}...")

            capacidad = extraer_capacidad_de_url(url, driver)

            # Si Selenium no encontro, intentar extraer del nombre en la URL
            if not capacidad:
                nombre_en_url = url.split("/pdp/")[-1].split("/")[0].replace("-", " ")
                capacidad = extraer_capacidad_de_nombre(nombre_en_url)

            print(f"  -> Capacidad: {capacidad if capacidad else 'No encontrada'}")

            resultados.append({
                "URL": url,
                "CodigoProducto": codigo,
                "SKU": sku,
                "Capacidad": capacidad
            })
    finally:
        driver.quit()

    df_capacidades = pd.DataFrame(resultados)

    # Guardar resultados
    df_capacidades.to_csv("liverpool_capacidades.csv", index=False)
    total = len(df_capacidades)
    con_cap = (df_capacidades["Capacidad"] != "").sum()
    print(f"\nTotal URLs procesadas: {total}")
    print(f"Con capacidad: {con_cap} | Sin capacidad: {total - con_cap}")
    print("Guardado en liverpool_capacidades.csv")
    print(df_capacidades.head(20))

URLs cargadas: 104
[1/104] SKU: 1199739693 | https://www.liverpool.com.mx/tienda/pdp/xiaomi-redmi-a7-pro-lcd-ips-6....
  -> Capacidad: 64 GB
[2/104] SKU: 1201368754 | https://www.liverpool.com.mx/tienda/pdp/xiaomi-redmi-note-15-amoled-6....
  -> Capacidad: 128 GB
[3/104] SKU: 1199887226 | https://www.liverpool.com.mx/tienda/pdp/motorola-edge-70-fusion-amoled...
  -> Capacidad: 256 GB
[4/104] SKU: 1201022467 | https://www.liverpool.com.mx/tienda/pdp/motorola-edge-60-pro-poled-6.6...
  -> Capacidad: 256 GB
[5/104] SKU: 1201894804 | https://www.liverpool.com.mx/tienda/pdp/honor-magic7-lite-amoled-6.78-...
  -> Capacidad: 512 GB
[6/104] SKU: 1201895894 | https://www.liverpool.com.mx/tienda/pdp/motorola-moto-g06-ips-6.8-pulg...
  -> Capacidad: 256 GB
[7/104] SKU: 1201712439 | https://www.liverpool.com.mx/tienda/pdp/xiaomi-redmi-note-15-pro-4g-am...
  -> Capacidad: 512 GB
[8/104] SKU: 1200341267 | https://www.liverpool.com.mx/tienda/pdp/xiaomi-redmi-note-15-pro-plus-...
  -> Capacidad: 256 G